# Input Generation — German Electricity Taxes & Levies 2026

## Purpose

This notebook defines and exports the **state-imposed electricity cost components** applied to residential consumers in Germany for 2026. All values are hardcoded from official 2026 publications — no raw data file is required.

**Output file:**  
- `inputs/residential_taxes_2026.csv` — per-component rates and totals (net and gross of VAT)

## Data sources

- Vattenfall. (2026, March). *Steuern und Umlagen 2026: Strom und Gas*. Retrieved April 15, 2026, from https://www.vattenfall.de/geschaeftskunden/ves/magazin/energie/preisentwicklung-strom-und-gas-2026
- KAV. *Konzessionsabgabenverordnung [KAV], §~2*. BGBl. I S. 12 (1992, January 9, as amended 2005, July 7). Retrieved April 15, 2026, from https://www.gesetze-im-internet.de/kav/__2.html

## Thesis reference

Chapter 3, Section 3.3 — *Retail cost structure*; Chapter 4, Section 4.2 — *Tax and levy parameterisation*

In [1]:
import os
from pathlib import Path
import pandas as pd

# ── Path configuration ─────────────────────────────────────────────────────────
def find_repo_root(marker='README.md'):
    p = Path(os.getcwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f'Repo root not found (looked for: {marker})')

REPO_ROOT  = find_repo_root()
INPUTS     = REPO_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = INPUTS / 'residential_taxes_2026.csv'

# ── Kategorie A component rates (ct/kWh, net of VAT) ──────────────────────────
# KWKG-Umlage: CHP surcharge, +61% vs 2025.
KWKG_UMLAGE        = 0.446
# §19 StromNEV-Umlage: Cat. A (≤ 1 GWh/year).
STROMNEV           = 1.559
# Offshore-Netzumlage: offshore grid expansion surcharge.
OFFSHORE_UMLAGE    = 0.941
# Konzessionsabgabe: KAV §2(2) representative average for standard customers.
# Actual rate varies by municipality (range 1.32–2.39 ct/kWh); 1.64 used as representative value.
KONZESSIONSABGABE  = 1.64
# Stromsteuer: federal electricity tax (StromStG §3), unchanged.
STROMSTEUER        = 2.05
# VAT rate (%).
VAT_PCT            = 19.0

TOTAL_NO_VAT       = round(KWKG_UMLAGE + STROMNEV + OFFSHORE_UMLAGE + KONZESSIONSABGABE + STROMSTEUER, 2)
TOTAL_INCL_VAT     = round(TOTAL_NO_VAT * (1 + VAT_PCT / 100), 2)

print(f'Total (net):   {TOTAL_NO_VAT:.3f} ct/kWh')
print(f'Total (gross): {TOTAL_INCL_VAT:.3f} ct/kWh  (incl. {VAT_PCT:.0f}% VAT)')

Total (net):   6.640 ct/kWh
Total (gross): 7.900 ct/kWh  (incl. 19% VAT)


## Step 1 — Component breakdown table

In [2]:
df_components = pd.DataFrame([
    ('KWKG-Umlage',          KWKG_UMLAGE,       'ct/kWh', 'CHP surcharge; +61% vs 2025'),
    ('§19 StromNEV-Umlage',  STROMNEV,          'ct/kWh', 'Cat. A (≤ 1 GWh/year)'),
    ('Offshore-Netzumlage',  OFFSHORE_UMLAGE,   'ct/kWh', 'Offshore grid expansion surcharge'),
    ('Konzessionsabgabe',    KONZESSIONSABGABE, 'ct/kWh', 'KAV §2(2) representative average; range 1.32–2.39 by municipality'),
    ('Stromsteuer',          STROMSTEUER,       'ct/kWh', 'Federal electricity tax (StromStG §3); unchanged'),
    ('Total (net of VAT)',   TOTAL_NO_VAT,      'ct/kWh', ''),
    ('VAT',                  VAT_PCT,           '%',      ''),
    ('Total (incl. VAT)',    TOTAL_INCL_VAT,    'ct/kWh', ''),
], columns=['component', 'value', 'unit', 'notes'])

display(df_components)

,component,value,unit,notes
0,KWKG-Umlage,0.446,ct/kWh,CHP surcharge; +61% vs 2025
1,§19 StromNEV-Umlage,1.559,ct/kWh,Cat. A (≤ 1 GWh/year)
2,Offshore-Netzumlage,0.941,ct/kWh,Offshore grid expansion surcharge
3,Konzessionsabgabe,1.640,ct/kWh,KAV §2(2) representative average; range 1.32–2...
4,Stromsteuer,2.050,ct/kWh,Federal electricity tax (StromStG §3); unchanged
5,Total (net of VAT),6.640,ct/kWh,
6,VAT,19.000,%,
7,Total (incl. VAT),7.900,ct/kWh,


## Step 2 — Export to CSV

In [3]:
df_export = pd.DataFrame([{
    'region':                      'DE',
    'KWKG_Umlage_ct_kWh':          KWKG_UMLAGE,
    'StromNEV_ct_kWh':             STROMNEV,
    'Offshore_Umlage_ct_kWh':      OFFSHORE_UMLAGE,
    'Konzessionsabgabe_ct_kWh':    KONZESSIONSABGABE,
    'Stromsteuer_ct_kWh':          STROMSTEUER,
    'Total_no_VAT_ct_kWh':         TOTAL_NO_VAT,
    'VAT_pct':                     VAT_PCT,
    'Total_incl_VAT_ct_kWh':       TOTAL_INCL_VAT,
}])

df_export.to_csv(OUTPUT_CSV, index=False)
print(f'Exported: {OUTPUT_CSV.name}')
display(df_export)

Exported: residential_taxes_2026.csv


,region,KWKG_Umlage_ct_kWh,StromNEV_ct_kWh,Offshore_Umlage_ct_kWh,Konzessionsabgabe_ct_kWh,Stromsteuer_ct_kWh,Total_no_VAT_ct_kWh,VAT_pct,Total_incl_VAT_ct_kWh
0,DE,0.446,1.559,0.941,1.64,2.05,6.64,19.0,7.9
